In [ ]:
#DQN Hyperparameter tuning
from Singe_ag_Environment_SparseR import SingleSatelliteEnvSR
from gymnasium.utils.env_checker import check_env
import traceback
print(type(SingleSatelliteEnvSR))
# This will catch many common issues
env = SingleSatelliteEnvSR("None")
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print("Environment has issues:")
    traceback.print_exc()

In [ ]:
import optuna

from cleanrl_utils.tuner import Tuner

dqntuner = Tuner(
    script="cleanrl/dqn.py",
    metric="charts/episodic_return",
    metric_last_n_average_window=50,
    direction="maximize",
    aggregation_type="average",
    target_scores={
        "SingleSatelliteEnvSR":None
    },
    params_fn=lambda trial: {
        "learning-rate": trial.suggest_float("learning-rate", 0.0003, 0.003, log=True),
        "buffer_size": trial.suggest_categorical("buffer_size", [10000, 100000, 1000000]),
        "tau": trial.suggest_float("tau",0,1),
        "target_network_frequency": trial.suggest_categorical("target_network_frequency", [500,1000, 1500,]),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32,64]),
        "start_e": trial.suggest_float("start_e", 0.5,1),
        "end_e":trial.suggest_float("end_e", 0,0.5),
        "exploration_fraction":trial.suggest_float("end_e", 0,0.5),
        "learning_starts":trial.suggest_int("learning_starts", 10000,1000000),
        "train_frequency":trial.suggest_int("train_frequency",1,10),
        "total-timesteps": 500000,
        "num-envs": 1,
    },
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    sampler=optuna.samplers.TPESampler(),
)
dqntuner.tune(
    num_trials=100,
    num_seeds=5,
)